# Ekstraksi Poligon Bangunan pada Permukiman Padat: demo hasil

**Mahathir Muhammad (6025252008)**, Pengolahan Citra Digital, S2 Informatika ITS.

Notebook ini memperlihatkan alur proyek dari citra sampai poligon pada beberapa *tile* asli, lalu menampilkan tabel dan grafik hasil.

- Notebook ini tipis: semua fungsi diimpor dari paket `densepoly`, dan sel di sini hanya memanggil, menjelaskan, dan memplot.
- Tidak ada training dan tidak ada angka yang dihitung ulang di sini. Tabel dan grafik **dibaca** dari keluaran skrip evaluasi
  (`$DENSEPOLY_OUT/tables` dan `$DENSEPOLY_OUT/figures`), dan model yang dipakai adalah *checkpoint* beku FULL-P0 *seed* 0.
- Waktu jalan dari atas ke bawah: di bawah 10 menit (GPU atau CPU).
- Training dari awal ada di `notebooks/full_pipeline.ipynb`.

In [ ]:
import os
import warnings
from dataclasses import replace
from pathlib import Path

warnings.filterwarnings("ignore")  # progress-bar and deprecation notices of the libraries

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, Markdown, display
from matplotlib.colors import hsv_to_rgb
from torch.utils.data import DataLoader, Subset

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
os.chdir(REPO)

from densepoly import infer, paths
from densepoly.classical import b0
from densepoly.config import load_config
from densepoly.data.dataset import TileDataset, tile_refs
from densepoly.labels.boundary_weight import boundary_mask
from densepoly.labels.instances import load_instances
from densepoly.labels.targets import targets_from_config
from densepoly.postproc.markers import foreground, markers
from densepoly.postproc.polygonize import PolygonParams, polygonize
from densepoly.priors.cache import read_rgb
from densepoly.priors.gap import gap_map
from densepoly.priors.mbi import brightness, mbi
from densepoly.utils.io import read_json

DATA, SPLIT, SEED = "crowdai", "test", 0
RUN = paths.run_name(SEED)
out = paths.out_root()
assert (out / "tables" / DATA / "ablation.md").exists(), f"no result tables in {out}: set DENSEPOLY_OUT to the outputs of the runs"
print("outputs:", out)

In [ ]:
# Small plotting helpers; every computation below is a call into densepoly.
GT_COLOUR, PREDICTION_COLOUR = "#39d353", "#ffd60a"
CLASS_COLOURS = np.array([[0, 0, 0], [200, 200, 200], [230, 40, 40]], np.uint8)


def show(panels, titles, width=3.0):
    """One row of images; a panel is an array or (array, imshow options)."""
    fig, axes = plt.subplots(1, len(panels), figsize=(width * len(panels), width + 0.3), squeeze=False)
    for ax, panel, title in zip(axes[0], panels, titles):
        array, style = panel if isinstance(panel, tuple) else (panel, {})
        ax.imshow(array, **{"interpolation": "nearest", **style})
        ax.set_title(title, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()


def outlined(image, instance_map):
    """The image with the boundaries of an instance map drawn in red."""
    shown = image.copy()
    shown[boundary_mask(instance_map) & (instance_map > 0)] = (255, 40, 40)
    return shown


def draw(ax, image, rings, colour, title):
    """Polygons, given as (n, 2) rings, over the image; the title counts them."""
    ax.imshow(image)
    for ring in rings:
        closed = np.vstack([ring, ring[:1]])
        ax.plot(closed[:, 0], closed[:, 1], color="black", linewidth=2.2, alpha=0.55)
        ax.plot(closed[:, 0], closed[:, 1], color=colour, linewidth=1.2)
    ax.set_xlim(0, image.shape[1])
    ax.set_ylim(image.shape[0], 0)
    ax.set_title(f"{title} ({len(rings)})", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


def theta_hue(sin4, cos4, valid):
    """θ = atan2(sin 4θ, cos 4θ) / 4 as a hue between 0° and 90°, black where it is not valid."""
    theta = (np.arctan2(sin4, cos4) / 4) % (np.pi / 2)
    hue = np.where(valid, theta / (np.pi / 2), 0.0).astype(np.float32)
    return hsv_to_rgb(np.dstack([hue, np.ones_like(hue), valid.astype(np.float32)]))


def rings_of(items, image_id):
    return [np.reshape(item["segmentation"][0], (-1, 2)) for item in items if item["image_id"] == image_id]

## 1. Masalah dan gambaran alur

Peta butuh **satu poligon rapi per rumah**, dengan sedikit verteks dan dinding lurus. Segmentasi biasa hanya memberi *mask* piksel,
dan di permukiman padat beberapa rumah yang atapnya berdempetan sering terbaca sebagai satu gumpalan. Metode terbaru mengatasinya
dengan belajar dari anotasi poligon, yang tidak selalu tersedia. Proyek ini hanya memakai *mask* per bangunan.

Alurnya: **citra RGB → U-Net++ (EfficientNet-B4) → tiga kelas (latar, bangunan, kontak) dan orientasi θ → marker dan *watershed* →
kontur per rumah → regularisasi poligon searah θ**.

Tiga gagasan diuji dengan aturan yang ditulis sebelum ada hasil: **N1** *head* orientasi θ yang labelnya diturunkan dari *mask*,
**N2** evaluasi per tingkat kepadatan pada data tanpa kebocoran, dan **N3** prior morfologi sebagai kanal input tambahan.

## 2. Data

CrowdAI punya banyak *tile* kembar: *tile* yang sama muncul lagi setelah diputar atau dicerminkan, juga di antara train dan val.
*Tile* kembar dikelompokkan dengan *perceptual hash*, lalu satu wakil per kelompok dibagi ke train, tuning, dan *test*.

In [ ]:
report = read_json(paths.splits_dir() / "dedup_report.json")
print(f"train asli        : {report['train']['images']:>7,} tile")
print(f"kelompok duplikat : {report['train']['clusters']:>7,}")
for name in ("train_dedup", "val_tune_sub", "val_select", "test", "val_clean"):
    print(f"{name:18s}: {report['splits'][name]:>7,} tile")

density = read_json(paths.density_file(DATA, SPLIT))
strata = density["main"]["strata"]
refs = {int(ref.key): ref for ref in tile_refs(DATA, SPLIT)}
rng = np.random.default_rng(0)
panels, titles = [], []
for stratum in ("jarang", "sedang", "padat"):
    image_id = int(rng.choice(sorted(strata[stratum])))
    instance_map = load_instances(DATA, SPLIT, image_id)
    panels.append(outlined(read_rgb(refs[image_id].image), instance_map))
    titles.append(f"{stratum}: {int(instance_map.max())} bangunan (tile {image_id})")
show(panels, titles, width=4.0)

Strata kepadatan (jarang, sedang, padat) adalah tertil jumlah bangunan per *tile*. Strata dihitung dari *ground truth* dan dibekukan
sebelum model apa pun dievaluasi. Garis merah di atas adalah batas bangunan pada *ground truth*.

*Tile* padat berikut dipakai di sisa notebook. Ketiganya diambil acak dengan *seed* tetap, sama dengan gambar di paper.

In [ ]:
dense = sorted(np.random.default_rng(0).choice(sorted(strata["padat"]), 3, replace=False).tolist())
images = {image_id: read_rgb(refs[image_id].image) for image_id in dense}
instance_maps = {image_id: load_instances(DATA, SPLIT, image_id) for image_id in dense}
gt_annotations = read_json(paths.gt_file(DATA, SPLIT))["annotations"]
gt_rings = {image_id: rings_of(gt_annotations, image_id) for image_id in dense}
touching = int(np.random.default_rng(0).choice(sorted(density["secondary"]["strata"]["bersentuhan"])))
print("tile padat:", dense, "| tile dengan bangunan bersentuhan:", touching)

## 3. Pengolahan citra klasik: *brightness*, MBI, peta celah, dan *baseline* B0

Sebelum *deep learning*, bangunan bisa dicari dengan morfologi matematika:

- ***Brightness***: nilai maksimum ketiga kanal warna.
- **MBI** (*Morphological Building Index*): respons *white top-hat* dengan elemen garis pada beberapa skala dan arah. Nilainya
  tinggi pada struktur terang seukuran rumah.
- **Peta celah**: *black top-hat* berarah. Nilainya tinggi pada garis gelap sempit, misalnya celah di antara dua atap.
- **B0**: MBI diambang (Otsu), dibersihkan dengan *opening*, dipisah dengan *watershed*, lalu dirapikan menjadi poligon.

In [ ]:
cfg_b0 = load_config(DATA, "b0")
b0_params = b0.B0Params.from_config(cfg_b0.postproc.b0)
b0_polygons = PolygonParams.from_config(cfg_b0.postproc)
for image_id in dense:
    image = images[image_id]
    b = brightness(image)
    mbi_map = mbi(b, list(cfg_b0.priors.mbi_scales_px))
    gap = gap_map(b, int(cfg_b0.priors.gap_se_length_px), str(cfg_b0.priors.gap_mode))
    found_b0 = b0.instances(infer.cached_mbi(DATA, SPLIT, image_id), b0_params)
    rings = [polygon.points for polygon in polygonize(found_b0.instance_map, b0_polygons)]
    fig, axes = plt.subplots(1, 5, figsize=(15, 3.3))
    for ax, array, title, style in (
        (axes[0], image, f"RGB (tile {image_id})", {}),
        (axes[1], b, "brightness", {"cmap": "gray"}),
        (axes[2], mbi_map, "MBI", {"cmap": "magma", "vmax": np.percentile(mbi_map, 99.5)}),
        (axes[3], gap, "peta celah", {"cmap": "magma", "vmax": np.percentile(gap, 99.5)}),
    ):
        ax.imshow(array, **style)
        ax.set_title(title, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    draw(axes[4], image, rings, PREDICTION_COLOUR, "poligon B0")
    plt.tight_layout()
    plt.show()

MBI memang menyala di atap yang terang, tetapi juga di jalan dan halaman yang terang, dan atap yang gelap terlewat. Itu sebabnya B0
hanya dipakai sebagai pembanding klasik (AP-nya mendekati nol, lihat tabel di bagian 7).

## 4. Label diturunkan dari *mask*

Jaringan dilatih dengan tiga jenis label, semuanya dibuat otomatis dari peta *instance* (satu nomor per bangunan):

- **Kelas**: latar, bangunan, dan **kontak**, yaitu piksel bangunan yang berjarak ≤ 2 piksel dari bangunan lain.
- **Orientasi θ**: arah sisi kotak terkecil (`minAreaRect`) tiap bangunan, modulo 90°. Targetnya (sin 4θ, cos 4θ), sehingga keempat
  sisi kotak memberi nilai yang sama.
- **Bobot tepi**: piksel dekat batas bangunan diberi bobot *loss* lebih besar.

In [ ]:
cfg = load_config(DATA, "full_p0")
alpha = float(cfg.labels.edge_weight_alpha)
for image_id in [*dense[:2], touching]:
    image, instance_map = read_rgb(refs[image_id].image), load_instances(DATA, SPLIT, image_id)
    targets = targets_from_config(instance_map, cfg.labels)
    show(
        [
            image,
            CLASS_COLOURS[targets.classes],
            theta_hue(targets.sin4, targets.cos4, targets.valid),
            (targets.weight, {"cmap": "viridis", "vmin": 1.0, "vmax": 1.0 + alpha}),
        ],
        [f"RGB (tile {image_id})", "kelas (abu: bangunan, merah: kontak)", "θ (warna = sudut 0°-90°)", "bobot tepi"],
        width=3.6,
    )
    share = (targets.classes == 2).sum() / max((targets.classes > 0).sum(), 1)
    print(f"tile {image_id}: piksel kontak = {100 * share:.3f}% dari piksel bangunan")

Baris terakhir adalah *tile* yang berisi bangunan bersentuhan; hanya di sana piksel kontak (merah) muncul. Piksel kontak sangat
sedikit, karena rumah yang benar-benar berdempetan jarang. Ini berakibat pada hasil: di sebagian besar model, kelas kontak tidak
pernah diprediksi (lihat bagian 5 dan 8).

## 5. Inferensi FULL-P0: dari keluaran jaringan sampai poligon

Sel berikut memuat *checkpoint* beku FULL-P0 dan menjalankannya pada ketiga *tile* padat. Keluarannya lima kanal: probabilitas latar,
bangunan, dan kontak, lalu (sin 4θ, cos 4θ).

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, selected = infer.load_network(cfg, paths.checkpoint_file(DATA, "full_p0", RUN), device)
tiles = TileDataset(cfg, SPLIT, "predict")
indexes = [i for i, unit in enumerate(tiles.units) if int(unit.key) in dense]
loader = DataLoader(Subset(tiles, indexes), batch_size=len(indexes))
channels = {
    image_id: tile.astype(np.float32)
    for image_id, tile, _ in infer.network_channels(model, loader, device, int(cfg.data.tile_size))
}
print("device:", device, "| channels per tile:", next(iter(channels.values())).shape)

In [ ]:
instance_params = infer.InstanceParams.from_config(cfg.postproc)  # the frozen tuning of FULL-P0
polygon_params = PolygonParams.from_config(cfg.postproc)
found, regular, raw = {}, {}, {}
for image_id in dense:
    p_bg, p_b, p_k, sin4, cos4 = channels[image_id]
    fg = foreground(p_bg)
    marker_labels = markers(p_b, p_k, fg, instance_params.tau, instance_params.marker_min_px)
    found[image_id] = infer.network_instances(channels[image_id], instance_params)
    regular[image_id] = infer.network_polygons(found[image_id], channels[image_id], polygon_params)
    raw[image_id] = polygonize(found[image_id].instance_map, replace(polygon_params, mode="raw"))
    show(
        [
            images[image_id],
            (p_b, {"cmap": "gray", "vmin": 0, "vmax": 1}),
            (p_k, {"cmap": "gray", "vmin": 0, "vmax": 1}),
            theta_hue(sin4, cos4, fg),
            (marker_labels > 0, {"cmap": "gray"}),
            (found[image_id].instance_map, {"cmap": "tab20"}),
        ],
        [f"RGB (tile {image_id})", "P bangunan", f"P kontak (maks {p_k.max():.3f})", "θ dari head", "marker",
         f"instance ({found[image_id].count})"],
        width=2.6,
    )

Peta **P kontak** hampir hitam: pada *checkpoint* ini kelas kontak tidak aktif, jadi marker pada dasarnya sama dengan *foreground*
dan rumah terpisah oleh celah latar yang diprediksi. Ini temuan yang dilaporkan apa adanya.

Berikut kontur mentah tiap *instance* (hanya disederhanakan dengan Douglas-Peucker) dibandingkan poligon hasil regularisasi searah θ.
Angka dalam kurung adalah jumlah poligon.

In [ ]:
for image_id in dense:
    fig, axes = plt.subplots(1, 3, figsize=(12.6, 4.4))
    draw(axes[0], images[image_id], gt_rings[image_id], GT_COLOUR, f"ground truth, tile {image_id}")
    draw(axes[1], images[image_id], [p.points for p in raw[image_id]], PREDICTION_COLOUR, "kontur mentah")
    draw(axes[2], images[image_id], [p.points for p in regular[image_id]], PREDICTION_COLOUR, "setelah regularisasi θ")
    plt.tight_layout()
    plt.show()
    vertices = [sum(len(p.points) for p in polygons) for polygons in (raw[image_id], regular[image_id])]
    boxes = sum(p.path == "box" for p in regular[image_id])
    print(f"tile {image_id}: verteks {vertices[0]} -> {vertices[1]}; {boxes} dari {len(regular[image_id])} poligon menjadi kotak;",
          f"ground truth {sum(len(r) for r in gt_rings[image_id])} verteks")

## 6. Beberapa model pada *tile* yang sama

Prediksi tersimpan (*seed* 0) dari empat model, digambar pada *tile* yang sama:

- **B4**: tiga kelas dan *watershed*, kontur mentah.
- **B5**: B4 dengan regularisasi memakai θ dari kotak terkecil tiap *instance*.
- **FULL-P0**: ditambah *head* θ yang memandu regularisasi.
- **B3m** (eksploratif): segmentasi biner biasa yang komponennya diregularisasi dengan cara B5.

In [ ]:
MODELS = [("b4", "B4"), ("b5", "B5"), ("full_p0", "FULL-P0"), ("b3m", "B3m")]
saved = {exp: read_json(paths.pred_file(DATA, exp, RUN, SPLIT)) for exp, _ in MODELS
         if paths.pred_file(DATA, exp, RUN, SPLIT).exists()}
for image_id in dense:
    fig, axes = plt.subplots(1, 1 + len(saved), figsize=(3.4 * (1 + len(saved)), 3.8))
    draw(axes[0], images[image_id], gt_rings[image_id], GT_COLOUR, f"ground truth, tile {image_id}")
    for ax, (exp, label) in zip(axes[1:], [m for m in MODELS if m[0] in saved]):
        draw(ax, images[image_id], rings_of(saved[exp], image_id), PREDICTION_COLOUR, label)
    plt.tight_layout()
    plt.show()

## 7. Tabel dan grafik hasil

Semua tabel di bawah dibuat `scripts/tables.py` dari file metrik dan hanya **dibaca** di sini. Selisih antar-model dihitung
berpasangan per *tile* dengan tiga *seed*, dan selang 95%-nya berasal dari *bootstrap* 1.000 replikasi.

### 7.1 Semua skenario di CrowdAI *test*

In [ ]:
def table(name, dataset=DATA):
    display(Markdown((out / "tables" / dataset / f"{name}.md").read_text(encoding="utf-8")))


table("ablation")

Cara membacanya: regularisasi (B4 → B5) yang membuat poligon rapi, terlihat dari C-IoU yang naik dan MTA yang turun. Model biner B3
punya AP tertinggi, sedangkan jalur tiga kelas (B4 dan seterusnya) paling sedikit menggabungkan rumah.

### 7.2 N1: *head* orientasi (didukung)

In [ ]:
table("n1")

### 7.3 N3: prior morfologi (tidak didukung)

In [ ]:
table("n3")

### 7.4 N2: efek per tingkat kepadatan

In [ ]:
table("n2")
for name in ("crowdai_n2_id.png", "crowdai_strata_id.png"):
    file = out / "figures" / "results" / name
    if file.exists():
        display(Image(filename=str(file), width=1000))

### 7.5 Banding dengan HiSup pada data tanpa kebocoran

In [ ]:
table("sota")

### 7.6 Eksploratif: *baseline* biner yang diregularisasi dan *seed* detektor

In [ ]:
table("exploratory")

### 7.7 Biaya komputasi

In [ ]:
table("efficiency")

## 8. Kesimpulan dan keterbatasan

**Yang didukung data**

- Poligon yang rapi bisa dihasilkan hanya dari *mask*: regularisasi geometris menaikkan C-IoU sekitar 10 poin.
- *Head* orientasi (N1) memperbaiki PoLiS dan MTA di *tile* padat. Efeknya kecil, tetapi searah di ketiga *seed* dan membesar
  seiring kepadatan.
- *Seed* dari detektor kotak miring, yang juga dilatih dari *mask*, mengurangi rumah yang tergabung dengan AP setara.

**Yang tidak didukung**

- Prior morfologi (N3) tidak mengurangi rumah yang tergabung.
- Kelas kontak tidak aktif di sebagian besar model, jadi mekanisme pemisah yang diadopsi hampir tidak bekerja.
- "Mendekati SOTA" tidak tercapai: pada *tile* val tanpa kembaran, FULL-P0 tertinggal lebih dari 3 AP dari HiSup.
- *Baseline* biner yang diregularisasi (B3m) lebih baik daripada FULL-P0 di semua metrik kecuali rumah tergabung.

**Keterbatasan**

- Banding dengan HiSup hanya memakai 212 *tile*, dan *tile* itu bukan sampel acak dari val.
- WHU hanya satu *seed*, dan daftar *tile* ujinya tidak persis sama dengan yang dipakai Pix2Poly.
- *Head* θ baru diuji di jalur tiga kelas, belum di jalur biner yang lebih kuat.
- Detektor dilatih sekali dan dipakai untuk ketiga *seed*; ambang *confidence*-nya terpilih di ujung grid.
- Poligon tidak punya lubang, dan deduplikasi berbasis *perceptual hash* bisa melewatkan kembaran yang tidak persis.